# Notebook 10.1 — Verification & Multi-Seed Ablation

This notebook replicates the original **Notebook 7.2** baseline exactly (using identical preprocessed arrays, MSE loss, and hyperparameters) to reconcile the performance discrepancy. It then runs key ablation experiments over 5 different random seeds to establish baseline validity and calculate statistical significance (mean ± std).

In [1]:
import math
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
# Load preprocessed arrays generated in Notebook 2 / used in Notebook 7.2
X_train = torch.tensor(np.load("pipeline/X_train.npy"), dtype=torch.float32)
y_train = torch.tensor(np.load("pipeline/y_train.npy"), dtype=torch.float32)

X_val = torch.tensor(np.load("pipeline/X_val.npy"), dtype=torch.float32)
y_val = torch.tensor(np.load("pipeline/y_val.npy"), dtype=torch.float32)

X_test = torch.tensor(np.load("pipeline/X_test.npy"), dtype=torch.float32)
y_test = torch.tensor(np.load("pipeline/y_test.npy"), dtype=torch.float32)

with open("pipeline/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Loaded data shapes:")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Loaded data shapes:
Train: torch.Size([1844, 60, 95]) | Val: torch.Size([348, 60, 95]) | Test: torch.Size([349, 60, 95])


In [3]:
FEATURE_GROUPS = {
    "price": ["Close","High","Low","Open","Previous_Close","Gap","Price_Change"],
    "returns": ["Daily_Return","Return","Log_Return","ROC_5","ROC_10"],
    "volatility": ["Rolling_Volatility","ATR","Historical_Volatility","Parkinson_Volatility","Garman_Klass","Rolling_STD","Rolling_Variance","Rolling_Return_STD"],
    "trend": ["EMA_9","EMA_21","EMA_50","EMA_200","EMA_12","EMA_26","MACD","Signal","MACD_Histogram"],
    "momentum": ["Momentum_5","Momentum_10","Momentum_20"],
    "volume": ["Volume","Volume_MA_5","Volume_MA_20","Relative_Volume","Volume_Change","Volume_Momentum","OBV","VWAP"],
    "candlestick": ["Body","Upper_Wick","Lower_Wick","Full_Range","Body_Ratio","Upper_Wick_Ratio","Lower_Wick_Ratio","Body_to_Wick","High_Low_Range","Open_Close_Range","True_Range"],
    "statistics": ["Rolling_Mean","Rolling_Min","Rolling_Max","Rolling_Median","Rolling_Skew","Rolling_Kurtosis","Rolling_Zscore","Rolling_Max_Return","Rolling_Min_Return"],
    "lags": ["Close_Lag_1","Return_Lag_1","Volume_Lag_1","Close_Lag_2","Return_Lag_2","Volume_Lag_2","Close_Lag_3","Return_Lag_3","Volume_Lag_3","Close_Lag_5","Return_Lag_5","Volume_Lag_5","Close_Lag_10","Return_Lag_10","Volume_Lag_10"],
    "breakout": ["Rolling_High_5","Rolling_Low_5","Rolling_High_10","Rolling_Low_10","Rolling_High_20","Rolling_Low_20","Distance_From_High_5","Distance_From_Low_5","Distance_From_High_10","Distance_From_Low_10","Distance_From_High_20","Distance_From_Low_20","Range_Position","Breakout_20","Breakdown_20"],
    "calendar": ["Day","Month","Quarter","DayOfWeek","WeekOfYear"]
}

FEATURE_INDICES = {
    group: [feature_columns.index(col) for col in cols]
    for group, cols in FEATURE_GROUPS.items()
}

BATCH_SIZE = 64
D_MODEL = 128
NUM_HEADS = 8
FF_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.15
REGIME_DIM = 32
GROUP_HIDDEN_DIM = 64

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 35
PATIENCE = 8

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MarketRegimeEncoder(nn.Module):
    def __init__(self, input_dim, feature_groups, hidden_dim=64, regime_dim=32):
        super().__init__()
        self.feature_groups = feature_groups
        self.group_embeddings = nn.ModuleDict({
            group: nn.Sequential(
                nn.Linear(len(indices), hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, regime_dim)
            )
            for group, indices in feature_groups.items()
        })
        self.temporal_score = nn.Linear(regime_dim, 1)
        self.fusion = nn.Sequential(
            nn.Linear(len(feature_groups) * regime_dim, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, regime_dim)
        )
    def forward(self, x):
        group_vectors = []
        for group, indices in self.feature_groups.items():
            group_sequence = x[:, :, indices]
            embedding = self.group_embeddings[group](group_sequence)
            scores = self.temporal_score(embedding)
            weights = torch.softmax(scores, dim=1)
            pooled = (weights * embedding).sum(dim=1)
            group_vectors.append(pooled)
        regime = torch.cat(group_vectors, dim=1)
        return self.fusion(regime)

class AdaptiveGateNetwork(nn.Module):
    def __init__(self, regime_dim, num_groups):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(regime_dim, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, num_groups)
        )
    def forward(self, regime):
        gates = self.network(regime)
        return torch.softmax(gates, dim=-1)

class AdaptiveFinancialContext(nn.Module):
    def __init__(self, d_head, feature_groups):
        super().__init__()
        self.feature_groups = feature_groups
        self.projections = nn.ModuleDict({
            group: nn.Linear(len(indices), d_head)
            for group, indices in feature_groups.items()
        })
    def forward(self, x, gates, use_gating=True):
        contexts = {}
        for i, (group, indices) in enumerate(self.feature_groups.items()):
            features = x[:, :, indices]
            embedding = self.projections[group](features)
            embedding = F.normalize(embedding, p=2, dim=-1)
            similarity = torch.matmul(embedding, embedding.transpose(-2, -1))
            if use_gating:
                contexts[group] = gates[:, i].view(-1, 1, 1) * similarity
            else:
                contexts[group] = (1.0 / len(self.feature_groups)) * similarity
        return contexts

class RelativeTemporalBias(nn.Module):
    def __init__(self, max_length, num_heads):
        super().__init__()
        self.max_length = max_length
        self.bias = nn.Parameter(torch.zeros(num_heads, 2 * max_length - 1))
        nn.init.trunc_normal_(self.bias, std=0.02)
    def forward(self, length):
        position = torch.arange(length, device=self.bias.device)
        relative = position[:, None] - position[None, :]
        relative += self.max_length - 1
        return self.bias[:, relative]

class ModularFinancialAttention(nn.Module):
    def __init__(self, d_head, num_heads, feature_groups, use_financial_context=True):
        super().__init__()
        self.scale = math.sqrt(d_head)
        self.use_financial_context = use_financial_context
        self.num_groups = len(feature_groups)
        
        if use_financial_context:
            self.context = AdaptiveFinancialContext(d_head, feature_groups)
            self.group_weights = nn.Parameter(torch.ones(self.num_groups))
            self.financial_weight = nn.Parameter(torch.tensor(0.5))
            
        self.temporal_bias = RelativeTemporalBias(100, num_heads)
        self.temporal_weight = nn.Parameter(torch.tensor(0.5))
    def forward(self, Q, K, V, raw_features, gates, use_gating=True):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if self.use_financial_context:
            contexts = self.context(raw_features, gates, use_gating)
            group_weights = torch.softmax(self.group_weights, dim=0)
            financial_bias = 0
            for i, matrix in enumerate(contexts.values()):
                financial_bias += group_weights[i] * matrix
            financial_bias = financial_bias.unsqueeze(1)
            financial_scale = torch.sigmoid(self.financial_weight)
            scores = scores + financial_scale * financial_bias
            
        temporal_bias = self.temporal_bias(Q.shape[-2]).unsqueeze(0)
        temporal_scale = torch.sigmoid(self.temporal_weight)
        scores = scores + temporal_scale * temporal_bias
        
        weights = torch.softmax(scores, dim=-1)
        return torch.matmul(weights, V), weights

class ModularMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, feature_groups, use_financial_context=True):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.attention = ModularFinancialAttention(self.head_dim, num_heads, feature_groups, use_financial_context)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x, raw_features, gates, use_gating=True):
        B, L, _ = x.shape
        Q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        out, weights = self.attention(Q, K, V, raw_features, gates, use_gating)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.out_proj(out), weights

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim, dropout=0.15):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
    def forward(self, x):
        return self.network(x)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, feature_groups, use_financial_context=True, dropout=0.15):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attention = ModularMultiHeadAttention(d_model, num_heads, feature_groups, use_financial_context)
        self.ffn = FeedForward(d_model, ff_dim, dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, raw_features, gates, use_gating=True):
        attn_out, weights = self.attention(self.norm1(x), raw_features, gates, use_gating)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, weights

class AblationAdaptiveTransformer(nn.Module):
    def __init__(self, input_dim, feature_groups, d_model=128, num_heads=8, ff_dim=256, num_layers=2,
                 use_gating=True, use_regime=True, use_financial_context=True, dropout=0.15):
        super().__init__()
        self.use_gating = use_gating
        self.use_regime = use_regime
        self.use_financial_context = use_financial_context
        self.feature_groups = feature_groups
        
        self.embedding = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, ff_dim, feature_groups, use_financial_context, dropout)
            for _ in range(num_layers)
        ])
        
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        
        if use_regime:
            self.market_encoder = MarketRegimeEncoder(input_dim, feature_groups)
            if use_gating:
                self.gate_network = AdaptiveGateNetwork(32, len(feature_groups))
    def forward(self, x):
        raw_features = x
        B, L, _ = x.shape
        
        # Gate setup
        if self.use_regime:
            regime = self.market_encoder(raw_features)
            if self.use_gating:
                gates = self.gate_network(regime)
            else:
                gates = torch.ones(B, len(self.feature_groups), device=x.device) / len(self.feature_groups)
        else:
            gates = torch.ones(B, len(self.feature_groups), device=x.device) / len(self.feature_groups)
            regime = None
            
        x = self.embedding(x)
        cls = self.cls_token.expand(B,-1,-1)
        x = torch.cat([cls, x], dim=1)
        
        cls_raw = torch.zeros(B, 1, raw_features.size(-1), device=x.device)
        raw_features = torch.cat([cls_raw, raw_features], dim=1)
        
        x = self.position(x)
        x = self.dropout(x)
        
        attention_maps = []
        for layer in self.layers:
            x, weights = layer(x, raw_features, gates, self.use_gating)
            attention_maps.append(weights)
            
        prediction = self.head(x[:, 0]).squeeze(-1)
        return prediction, attention_maps, regime, gates

In [5]:
def calculate_trading_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    predicted_direction = np.sign(predicted)
    predicted_direction[np.abs(predicted) < 1e-6] = 0.0
    
    directional_accuracy = np.mean(np.sign(actual) == predicted_direction)
    hit_rate = directional_accuracy
    
    # De-biased backtest returns: non-overlapping returns (every 5 steps)
    strategy_returns = predicted_direction * actual
    mean_ret = np.mean(strategy_returns)
    std_ret = np.std(strategy_returns) + 1e-9
    
    # Annualized Sharpe ratio corrected for 5-day horizon
    sharpe = (mean_ret / std_ret) * np.sqrt(252 / 5)
    
    non_overlapping_returns = strategy_returns[::5]
    cumulative_return = np.cumprod(1 + non_overlapping_returns)[-1] - 1
    
    cum_series = np.cumprod(1 + non_overlapping_returns)
    running_max = np.maximum.accumulate(cum_series)
    drawdown = (cum_series - running_max) / (running_max + 1e-8)
    max_drawdown = drawdown.min()
    
    return {
        "Directional Accuracy": directional_accuracy,
        "Hit Rate": hit_rate,
        "Sharpe": sharpe,
        "Strategy Return": cumulative_return,
        "Max Drawdown": max_drawdown
    }

In [6]:
def run_seed_experiment(seed, X_tr, y_tr, X_va, y_va, X_te, y_te, feature_groups,
                        use_regime=True, use_gating=True, use_financial_context=True):
    # Set random seeds exactly
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    
    input_dim = X_tr.shape[2]
    model = AblationAdaptiveTransformer(
        input_dim=input_dim, feature_groups=feature_groups, d_model=D_MODEL, 
        num_heads=NUM_HEADS, ff_dim=FF_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
        use_gating=use_gating, use_regime=use_regime, use_financial_context=use_financial_context
    ).to(DEVICE)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds, _, _, _ = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                preds, _, _, _ = model(X_batch)
                val_loss += criterion(preds, y_batch).item()
        val_loss /= len(val_loader)
        
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            break
            
    # Test set evaluation
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_model_state.items()})
    model.eval()
    with torch.no_grad():
        preds, _, _, _ = model(X_te.to(DEVICE))
        preds_np = preds.cpu().numpy()
        
    y_te_np = y_te.numpy()
    mae = mean_absolute_error(y_te_np, preds_np)
    rmse = np.sqrt(mean_squared_error(y_te_np, preds_np))
    r2 = r2_score(y_te_np, preds_np)
    
    trade_metrics = calculate_trading_metrics(y_te_np, preds_np)
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Directional Accuracy": trade_metrics["Directional Accuracy"],
        "Sharpe": trade_metrics["Sharpe"],
        "Strategy Return": trade_metrics["Strategy Return"],
        "Max Drawdown": trade_metrics["Max Drawdown"]
    }

In [7]:
print("Selecting Top 40 features using Random Forest...")
df_rf = pd.read_parquet("processed_data/master_dataset.parquet")
all_features = []
for cols in FEATURE_GROUPS.values():
    all_features.extend(cols)

X_rf = df_rf[all_features].fillna(0)
y_rf = df_rf["Future_Return"].fillna(0)

rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_rf.iloc[:int(len(X_rf)*0.7)], y_rf.iloc[:int(len(X_rf)*0.7)])

importances = pd.Series(rf.feature_importances_, index=all_features)
top_40_features = importances.sort_values(ascending=False).head(40).index.tolist()

top_40_indices = [feature_columns.index(f) for f in top_40_features]
print("Top 5 selected features:", top_40_features[:5])
print("Top 40 indices compiled successfully.")

Selecting Top 40 features using Random Forest...


Top 5 selected features: ['Rolling_Skew', 'EMA_200', 'Signal', 'Parkinson_Volatility', 'MACD_Histogram']
Top 40 indices compiled successfully.


In [8]:
SEEDS = [42, 101, 2023, 777, 999]
experiments_results = {}

# 1. Baseline AFT (95 features)
print("\n--- Running Baseline AFT (95 features) over 5 seeds ---")
baseline_runs = []
for s in SEEDS:
    t0 = time.time()
    res = run_seed_experiment(s, X_train, y_train, X_val, y_val, X_test, y_test, FEATURE_INDICES)
    baseline_runs.append(res)
    print(f"  Seed {s} -> R2: {res['R2']:.4f} | MAE: {res['MAE']:.4f} | Sharpe: {res['Sharpe']:.4f} (took {time.time()-t0:.1f}s)")
experiments_results["Baseline AFT"] = baseline_runs

# 2. No Market Regime Encoder (95 features)
print("\n--- Running No Market Regime Encoder (95 features) over 5 seeds ---")
no_regime_runs = []
for s in SEEDS:
    t0 = time.time()
    res = run_seed_experiment(s, X_train, y_train, X_val, y_val, X_test, y_test, FEATURE_INDICES,
                              use_regime=False, use_gating=False)
    no_regime_runs.append(res)
    print(f"  Seed {s} -> R2: {res['R2']:.4f} | MAE: {res['MAE']:.4f} | Sharpe: {res['Sharpe']:.4f} (took {time.time()-t0:.1f}s)")
experiments_results["No Market Regime Encoder"] = no_regime_runs

# 3. Top-40 Features (No Regime Encoder)
print("\n--- Running Top-40 Features (No Regime Encoder) over 5 seeds ---")
top_groups = {"top": list(range(0, 40))}
X_tr_top = X_train[:, :, top_40_indices]
X_va_top = X_val[:, :, top_40_indices]
X_te_top = X_test[:, :, top_40_indices]

top40_runs = []
for s in SEEDS:
    t0 = time.time()
    res = run_seed_experiment(s, X_tr_top, y_train, X_va_top, y_val, X_te_top, y_test, top_groups,
                              use_regime=False, use_gating=False)
    top40_runs.append(res)
    print(f"  Seed {s} -> R2: {res['R2']:.4f} | MAE: {res['MAE']:.4f} | Sharpe: {res['Sharpe']:.4f} (took {time.time()-t0:.1f}s)")
experiments_results["Top-40 Features"] = top40_runs


--- Running Baseline AFT (95 features) over 5 seeds ---


  Seed 42 -> R2: -0.1193 | MAE: 0.0326 | Sharpe: 0.1672 (took 20.2s)


  Seed 101 -> R2: -0.0307 | MAE: 0.0310 | Sharpe: 0.7839 (took 17.0s)


  Seed 2023 -> R2: -0.0941 | MAE: 0.0327 | Sharpe: -0.7286 (took 14.2s)


  Seed 777 -> R2: -0.0267 | MAE: 0.0311 | Sharpe: 0.0103 (took 18.1s)


...

  Seed 42 -> R2: -0.0921 | MAE: 0.0323 | Sharpe: 0.3205 (took 4.9s)


  Seed 101 -> R2: -0.0091 | MAE: 0.0306 | Sharpe: 0.7839 (took 16.3s)


  Seed 2023 -> R2: -0.1085 | MAE: 0.0325 | Sharpe: 0.2992 (took 15.2s)


  Seed 777 -> R2: -0.0343 | MAE: 0.0309 | Sharpe: 0.1178 (took 12.4s)


  Seed 999 -> R2: -0.0644 | MAE: 0.0319 | Sharpe: -0.3366 (took 12.3s)


In [9]:
summary_rows = []
for exp_name, runs in experiments_results.items():
    df_runs = pd.DataFrame(runs)
    means = df_runs.mean()
    stds = df_runs.std()
    
    summary_rows.append({
        "Experiment": exp_name,
        "MAE": f"{means['MAE']:.4f} ± {stds['MAE']:.4f}",
        "RMSE": f"{means['RMSE']:.4f} ± {stds['RMSE']:.4f}",
        "R2": f"{means['R2']:.4f} ± {stds['R2']:.4f}",
        "Directional Accuracy": f"{means['Directional Accuracy']:.4f} ± {stds['Directional Accuracy']:.4f}",
        "Sharpe": f"{means['Sharpe']:.4f} ± {stds['Sharpe']:.4f}",
        "Strategy Return": f"{means['Strategy Return']:.4f} ± {stds['Strategy Return']:.4f}",
        "Max Drawdown": f"{means['Max Drawdown']:.4f} ± {stds['Max Drawdown']:.4f}"
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("experiments/verification_results.csv", index=False)
print("\n================ SUMMARY STATISTICS OVER 5 SEEDS ================")
print(df_summary.to_string(index=False))


================ SUMMARY STATISTICS OVER 5 SEEDS ================
              Experiment             MAE            RMSE               R2 Directional Accuracy           Sharpe  Strategy Return     Max Drawdown
            Baseline AFT 0.0316 ± 0.0010 0.0434 ± 0.0009 -0.0576 ± 0.0459      0.5284 ± 0.0575  0.2423 ± 0.6778  0.0579 ± 0.2537 -0.3202 ± 0.0500
No Market Regime Encoder 0.0321 ± 0.0011 0.0438 ± 0.0009 -0.0777 ± 0.0459      0.4779 ± 0.0393 -0.3936 ± 0.4399 -0.2124 ± 0.2388 -0.3776 ± 0.1315
         Top-40 Features 0.0316 ± 0.0009 0.0434 ± 0.0008 -0.0617 ± 0.0407      0.5266 ± 0.0328  0.2370 ± 0.4041 -0.0055 ± 0.2030 -0.2939 ± 0.0602
